In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!pip install -q pydantic==2.12.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 8.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [3]:
!pip install -q rwkv==0.8.31

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.9/410.9 kB 7.4 MB/s eta 0:00:00a 0:00:01


In [29]:
!pip install -qi https://test.pypi.org/simple/ rwkvx==0.2.0.dev14

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.1/427.1 kB 7.7 MB/s eta 0:00:00a 0:00:01


In [5]:
!pip install -q rwkv huggingface_hub

In [6]:
import os

import huggingface_hub

In [7]:
os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

In [30]:
from rwkvx.generation import (
    RWKVSession,
    RWKVTokenizer,
    RWKVModel,
    RWKVSampler,
    GenerationConfig
)

from rwkvx.conversation import (
    History,
    ConversationManager,
    Message,
    RWKV7PromptRenderer,
    get_default_rwkv7_prompt_templates,
)

In [9]:
model_title = "rwkv7-g1b-2.9b-20251205-ctx8192"
# model_title = "rwkv7-g0b-7.2b-20251220-ctx8192"
# model_title = "rwkv7-g0b-13.3b-20251130-ctx8192"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")

rwkv7-g1b-2.9b-20251205-ctx8192.pth:   0%|          | 0.00/5.90G [00:00<?, ?B/s]

In [10]:
config = GenerationConfig(
    max_new_tokens=1000,
    stop_tokens=[0, 261],  # 0, \n\n
)

print(config)

max_new_tokens=1000 temperature=1.0 top_p=0.3 top_k=0 presence_penalty=0.5 frequency_penalty=0.5 decay_penalty=0.996 stop_tokens=[0, 261]


In [21]:
model = RWKVModel(model_path=model_path, strategy='cuda bf16')

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/c4d4f4748a8233e9a2f1ee7f38f4c4e3780e750d/rwkv7-g1b-2.9b-20251205-ctx8192 (cuda bf16)



In [31]:
tokenizer = RWKVTokenizer()
sampler = RWKVSampler()

In [23]:
templates = get_default_rwkv7_prompt_templates()
renderer = RWKV7PromptRenderer(templates)

mgr = ConversationManager(renderer)

In [32]:
session = RWKVSession(model, tokenizer, sampler, mgr, config)

In [33]:
session.reset()

In [34]:
mgr.add_system('Include emojis to all your answers.')
mgr.build_prompt()

'System: Include emojis to all your answers.'

In [35]:
mgr.add_user('Tell me a short joke.')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think'

In [36]:
joke2 = session.generate_message()
print(joke2)

role='assistant' text='\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [37]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂'

In [38]:
mgr.add_user('Tell me a joke different from previous one.')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think'

In [39]:
joke = session.generate_message()
print(joke)

role='assistant' text='\n😄 Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [40]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂'

In [41]:
mgr.add_user('Tell me more complex joke.')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think'

In [42]:
joke3 = session.generate_message()
print(joke3)

role='assistant' text='\n😄 Here’s a more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions. 📚😭' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [43]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions. 📚😭'

In [44]:
mgr.add_user('List topics of all jokes that you have told me. Explain the second joke.', metadata={'think_mode': ' think'})
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions. 📚😭\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think'

In [45]:
final_answer = session.generate_message()

In [46]:
print(final_answer.metadata['think_text'])

We need to list all jokes told so far, and explain the second joke (the one about tomato). The user wants topics of all jokes told. So we need to compile a list of jokes, with topic categories if possible. Also explain the second joke.
We have three jokes:
1. "Why did the scarecrow win an award?" - Explanation: Scarecrow won an award because he was outstanding in his field (corn). Topic: Agriculture/farming, awards, puns.
2. "Why did the tomato turn red?" - Explanation: Because it saw the salad dressing. Topic: Food, color change, puns on "red".
3. "Why did the math book look sad?" - Explanation: Because it had too many problems and no solutions. Topic: Math, sadness, puns on "problems" and "solutions".
We need to present them clearly. Also note that each joke is a pun or wordplay. Provide explanation for each.
Make sure to answer in a friendly tone.


In [47]:
print(final_answer.text)

### All Jokes I’ve Shared So Far  
| # | Joke | Explanation (what makes it funny) |
|---|------|-----------------------------------|
| 1 | **“Why did the scarecrow win an award?”** | The punchline “because he was outstanding in his field” plays on two meanings of “field”: the literal corn‑field where scarecrows work and the figurative “award‑winning” category (e.g., Best Corn Harvest). It’s a classic **agricultural‑themed pun** that works because farmers and gardeners often use both senses of “field.” |
| 2 | **“Why did the tomato turn red?”** | The tomato turns red because it sees the **salad dressing**—a play on words (“red” as color and “dressing” as condiment). It’s a simple **food‑color pun** that most people can picture instantly, making it easy to remember and share. |
| 3 | **“Why did the math book look sad?”** | The book is sad because it has too many **math problems** (the “problems”) but no solutions (“solutions”). This creates a double‑meaning twist: sadness from being over

In [48]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions. 📚😭\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think>We need to list all jokes told so far, and explain the second joke (the one about tomato). The user wants topics of all jokes told. So we need to compile a list of jokes, with topic categories if possible. Also explain the second joke.\nWe have three jokes:\n1. "Why 

In [49]:
mgr.add_user('Lets wrap up. Thank you!')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions. 📚😭\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think>We need to list all jokes told so far, and explain the second joke (the one about tomato). The user wants topics of all jokes told. So we need to compile a list of jokes, with topic categories if possible. Also explain the second joke.\nWe have three jokes:\n1. "Why 

In [50]:
session.generate_message()

Message(role='assistant', text="\nYou're very welcome! 🎉 It was great chatting with you—hope you enjoyed the jokes and maybe even learned something new along the way! If you ever want more laughs or help with anything else, just let me know. Have a fantastic day! 😊✨", metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'})

In [51]:
session._last_processed_index

10

In [52]:
len(session.manager.history)

11